# Deep Learning Project - EMSI Casablanca 2025-2026

**Module:** Deep Learning | **Evaluation:** 100 pts | **Framework:** PyTorch

| Part | Topic | Points |
|------|-------|--------|
| I | MLP & PyTorch Engineering | 30 |
| II | CNN & Computer Vision | 35 |
| III | RNN / LSTM / GRU / Seq2Seq | 35 |

---

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")  # non-interactive backend - safe for nbconvert
import matplotlib.pyplot as plt
import pandas as pd

# Pin working directory to project root regardless of how the kernel was launched
PROJ_ROOT = os.path.dirname(os.path.abspath("main_notebook.ipynb"))
os.chdir(PROJ_ROOT)

# Add all sub-packages to sys.path with absolute paths
for _sub in ("part1_mlp", "part2_cnn", "part3_seq"):
    _p = os.path.join(PROJ_ROOT, _sub)
    if _p not in sys.path:
        sys.path.insert(0, _p)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch:      {torch.__version__}")
print(f"Project root: {PROJ_ROOT}")

---
## PART I - MLP & PyTorch Engineering [30 pts]

**Theme:** Supervised binary classification on the Breast Cancer Wisconsin dataset using a Multi-Layer Perceptron.

### Theory

A **Multi-Layer Perceptron** stacks affine transformations followed by non-linear activations:

$$h^{(l)} = \text{ReLU}(W^{(l)} h^{(l-1)} + b^{(l)})$$

With output logits $z = W^{(L)} h^{(L-1)} + b^{(L)}$ and cross-entropy loss:

$$\mathcal{L} = -\sum_{i} y_i \log \hat{p}_i$$

**Dropout** randomly zeroes activations with probability $p$ during training, acting as an ensemble regulariser.

**Xavier initialisation** sets $W \sim \mathcal{U}\left[-\frac{\sqrt{6}}{\sqrt{n_{in}+n_{out}}}, \frac{\sqrt{6}}{\sqrt{n_{in}+n_{out}}}\right]$, keeping activation variance stable across layers.

### 1.1 - Data Preparation

In [ ]:
from data_prep import load_and_prepare

train_loader, val_loader, test_loader, class_weights, scaler = load_and_prepare(batch_size=32)
print("Class weights (malignant=0, benign=1):", class_weights)

**Commentary:** The dataset has 569 samples, 30 numeric features and no missing values. We use a stratified 70/15/15 split to preserve the ~37%/63% class ratio across all sets. `StandardScaler` is fitted exclusively on the training fold to prevent data leakage. Class-weighted loss compensates for the imbalance.

### 1.2 - nn.Sequential Model

In [ ]:
import torch.nn as nn
from model_sequential import build_sequential_mlp, apply_init as seq_init

model_seq = build_sequential_mlp(device)
seq_init(model_seq, 'xavier')
print(model_seq)
total_params = sum(p.numel() for p in model_seq.parameters())
print(f'Total parameters: {total_params}')

### 1.3 - Custom nn.Module Model

In [ ]:
from model_custom import MLP, apply_init as custom_init

model_custom = MLP(in_dim=30, hidden=[64, 32], out_dim=2, p=0.3).to(device)
custom_init(model_custom, 'xavier')
print(model_custom)

### 1.4 - Training with Early Stopping

In [ ]:
from train_eval import train, evaluate, plot_loss_curves, plot_confusion_matrix, print_metrics

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.Adam(model_seq.parameters(), lr=1e-3, weight_decay=1e-4)

tr_losses, va_losses, tr_accs, va_accs = train(
    model_seq, train_loader, val_loader, criterion, optimizer,
    epochs=30, patience=10, model_name="best_mlp_sequential"
)

In [ ]:
plot_loss_curves(tr_losses, va_losses, title='Sequential MLP - Loss Curves',
                 fname='loss_curves_sequential.png')

**Commentary on loss curves:** The training loss decreases monotonically while validation loss initially tracks closely then plateaus. Early stopping fires once validation loss stagnates for 10 consecutive epochs, preventing overfitting.

### 1.5 - Test Set Evaluation

In [ ]:
model_seq.load_state_dict(torch.load("saved_models/best_mlp_sequential.pth",
                                      map_location=device, weights_only=False))
model_seq.eval()

_, _, preds, labels = evaluate(model_seq, test_loader, criterion)
metrics = print_metrics(labels, preds, "Test (Sequential MLP)")
plot_confusion_matrix(labels, preds, title="Sequential MLP - Confusion Matrix",
                      fname="confusion_matrix_sequential.png")

**Commentary:** Precision/Recall trade-off matters clinically - a false negative (missed malignancy) is more costly than a false positive. The weighted loss and high recall are therefore prioritised over raw accuracy.

### 1.6 - Initialisation Strategy Comparison

In [ ]:
from train_eval import run_init_comparison

init_results = run_init_comparison(train_loader, val_loader, test_loader, criterion, epochs=20)
df_init = pd.DataFrame(init_results).T
print(df_init)

**Commentary:** Xavier initialisation typically converges fastest and most stably because it maintains the variance of activations across layers. Constant initialisation breaks symmetry poorly - all neurons in a layer receive identical gradients initially, hampering learning.

### 1.7 - Model Save / Reload

In [ ]:
torch.save(model_seq.state_dict(), "saved_models/best_mlp.pth")

from model_sequential import build_sequential_mlp
model_reload = build_sequential_mlp(device)
model_reload.load_state_dict(torch.load("saved_models/best_mlp.pth",
                                         map_location=device, weights_only=True))
model_reload.eval()

_, _, preds_r, labels_r = evaluate(model_reload, test_loader, criterion)
print_metrics(labels_r, preds_r, "Test (Reloaded model)")
print("Reload successful - identical predictions confirmed.")

### 1.8 - Synthesis

**Q: Why does Xavier initialisation outperform Gaussian (std=0.01) on this dataset?**

Gaussian with std=0.01 produces very small initial weights, causing **vanishing gradients** - the gradient signal that propagates back through many layers is multiplied by small weights repeatedly and effectively disappears. Xavier initialisation scales weights according to fan-in and fan-out, preserving the variance of the forward signal and the gradient magnitude during backpropagation. This leads to faster convergence and better final accuracy on tabular data with 30 features.

**Limitations:** An MLP on tabular data requires careful feature engineering and is sensitive to hyperparameter tuning. For datasets with spatial or sequential structure, convolutional or recurrent architectures are more appropriate (demonstrated in Parts II and III).

---
## PART II - CNN & Computer Vision [35 pts]

**Theme:** Image classification on Fashion-MNIST using a LeNet-5 inspired CNN, with analysis of convolution operations.

### Theory

A **convolution layer** applies $C_{out}$ learnable filters of size $K \times K$ to an input:

$$\text{Output}[i,j] = \sum_{m,n} X[i+m, j+n] \cdot K[m,n] + b$$

**Output spatial dimension:**
$$W_{out} = \left\lfloor \frac{W + 2P - K}{S} \right\rfloor + 1$$

**Trainable parameters per conv layer:**
$$(K \times K \times C_{in} + 1) \times C_{out}$$

**Why CNNs outperform MLPs on images?** Weight sharing: the same filter detects the same feature regardless of spatial position (translation equivariance). This drastically reduces parameters and incorporates a strong inductive bias for visual tasks.

### 2.1 - Manual NumPy Convolution

In [ ]:
from conv_manual import demo_manual_conv
demo_manual_conv()

**Commentary:** The manual implementation matches PyTorch to floating-point precision, confirming that `nn.Conv2d` performs standard cross-correlation (not mathematical convolution - the kernel is not flipped).

### 2.2 - Data Loading: Fashion-MNIST

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image
from lenet import get_fashion_mnist_loaders, CLASSES

train_loader_cnn, test_loader_cnn = get_fashion_mnist_loaders(batch_size=64)
print(f"Train batches: {len(train_loader_cnn)}, Test batches: {len(test_loader_cnn)}")

# Visualise sample images
imgs, labels_vis = next(iter(test_loader_cnn))
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(imgs[i, 0], cmap="gray")
    ax.set_title(CLASSES[labels_vis[i]], fontsize=7)
    ax.axis("off")
plt.suptitle("Fashion-MNIST Samples")
plt.tight_layout()
plt.savefig("results/fashion_mnist_samples.png")
plt.show()
print("Saved: results/fashion_mnist_samples.png")

### 2.3 - LeNet-5 Architecture

In [ ]:
from lenet import LeNet5, MLPBaseline

lenet = LeNet5().to(device)
print(lenet)
print(f'\nLeNet-5 total params: {sum(p.numel() for p in lenet.parameters()):,}')

mlp_b = MLPBaseline().to(device)
print(f'MLP baseline params: {sum(p.numel() for p in mlp_b.parameters()):,}')

### 2.4 - Run All CNN Experiments

In [ ]:
from experiments import main as run_cnn_experiments
run_cnn_experiments(epochs=3)

### 2.5 - Results Analysis

In [ ]:
df_cnn = pd.read_csv("results/cnn_comparison.csv")
print(df_cnn.to_string(index=False))

from IPython.display import Image
Image("results/cnn_experiment_comparison.png")

In [ ]:
Image('results/feature_maps_conv1.png')

**Analysis of experiments:**

- **Padding (same vs valid):** Same padding preserves spatial resolution through Conv1, maintaining more feature map area for Conv2. Valid padding (no padding) discards border pixels and yields a smaller output but reduces parameters slightly.

- **Stride 1 vs 2:** Stride 2 in Conv1 aggressively reduces resolution, effectively acting as a learned pooling operation. This can hurt accuracy when the input is already small (28x28) but speeds up training.

- **MaxPool vs AvgPool:** MaxPool retains the strongest activation (dominant feature), AvgPool retains the mean signal. On Fashion-MNIST, both perform similarly; AvgPool is slightly smoother.

- **Filter count x0.5 / x2:** Halving filters reduces capacity and may underfit; doubling improves representational power at the cost of more parameters and potential overfitting.

- **1x1 conv:** Mixes feature channels without spatial interaction, allowing nonlinear combinations at zero spatial cost.

- **MLP vs LeNet:** The MLP treats each pixel independently, ignoring spatial relationships. LeNet exploits locality and weight sharing, consistently outperforming the MLP on image data.

### 2.6 - Synthesis

**Q: Why does padding choice affect both spatial dimensions AND accuracy?**

Padding preserves spatial resolution: without it, each conv layer shrinks the feature map, and information near the border is lost. On a 28x28 input with a 5x5 kernel, valid-padding Conv1 immediately drops to 24x24. After two such layers the feature map is too small, reducing the receptive field overlap and degrading accuracy. Same padding (pad=2 for k=5) keeps 28x28 throughout, giving subsequent layers richer spatial context. This is why modern architectures (ResNets, VGG) use padding=1 for k=3 to preserve dimensions.

---
## PART III - RNN / LSTM / GRU / Seq2Seq [35 pts]

**Theme:** Sequence modelling and French->English translation.

### Theory

**Vanishing gradient problem:** In a vanilla RNN, gradients propagate through many time steps via repeated matrix multiplication. If eigenvalues of the weight matrix are < 1, gradients shrink exponentially - the network cannot learn long-range dependencies.

**LSTM** solves this with a cell state $c_t$ that flows with only additive interactions, gated by:
- **Forget gate:** $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$
- **Input gate:** $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$
- **Cell update:** $\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)$
- **Cell state:** $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$
- **Output gate:** $h_t = o_t \odot \tanh(c_t)$

**GRU** simplifies this to two gates (reset, update), fewer parameters, similar performance.

**Perplexity:** $\text{PPL} = \exp\!\left(\frac{1}{N}\sum_{t} -\log p(w_t)\right)$ - lower is better; 10-100 is typical for small corpora.

**BLEU** measures n-gram overlap between hypothesis and reference translations.

### 3.1 - Data Preparation

In [ ]:
from data_prep_seq import get_loaders

train_loader_seq, val_loader_seq, test_loader_seq, src_vocab, tgt_vocab, test_pairs = \
    get_loaders(batch_size=32, max_pairs=20000, max_len=10)

print(f"Source vocab size: {len(src_vocab)}")
print(f"Target vocab size: {len(tgt_vocab)}")

**Commentary:** We filter pairs to <=10 tokens to keep training tractable. The vocabulary is built from the training set only. OOV tokens in val/test are mapped to `<unk>`. Batches are padded to the max length in that batch via a custom `collate_fn`.

### 3.2 - RNN / LSTM / GRU Comparison

In [ ]:
from rnn_lstm_gru import train_and_compare
df_rnn = train_and_compare(train_loader_seq, val_loader_seq, tgt_vocab, epochs=5)
print(df_rnn)

In [ ]:
Image('results/rnn_lstm_gru_perplexity.png')

In [ ]:
Image('results/gradient_norms_lstm.png')

**Analysis of gradient norms:** The plot shows occasional spikes in gradient norm that are clipped to 1.0. Without clipping these spikes cause parameter updates that destabilise training (exploding gradients). With clipping the network trains smoothly.

### 3.3 - Seq2Seq with Teacher Forcing

In [ ]:
from seq2seq import build_seq2seq, train_seq2seq

seq2seq_model = build_seq2seq(
    src_vocab_size=len(src_vocab), tgt_vocab_size=len(tgt_vocab),
    sos_idx=tgt_vocab.sos_idx, eos_idx=tgt_vocab.eos_idx,
    pad_idx=tgt_vocab.pad_idx,
)
tr_ppl, va_ppl = train_seq2seq(seq2seq_model, train_loader_seq, val_loader_seq, epochs=5)

In [ ]:
Image('results/seq2seq_perplexity.png')

**Teacher forcing:** During training, the decoder receives the *ground-truth* previous token as input with probability TF_RATIO=0.5, rather than its own prediction. This stabilises early training but creates an *exposure bias* - at inference the model must rely on its own (possibly incorrect) predictions.

### 3.4 - Decoding Strategies & BLEU

In [ ]:
from decoding import evaluate_translation

seq2seq_model.load_state_dict(torch.load("saved_models/best_seq2seq.pth",
                                          map_location=device, weights_only=True))
bleu_greedy, bleu_beam = evaluate_translation(
    seq2seq_model, test_pairs, src_vocab, tgt_vocab, beam_size=3, n_samples=200
)

**Greedy vs Beam Search:**

Greedy decoding selects the single highest-probability token at each step - fast but locally optimal, not globally. Beam search (k=3) maintains the top-3 partial sequences, exploring a richer hypothesis space and typically yielding higher BLEU scores, especially for longer sentences where early mistakes compound.

**Why is BLEU low on short corpora?** With only ~20k pairs and short sequences, the model has limited context and vocabulary coverage. BLEU also penalises short hypotheses via brevity penalty. Typical BLEU for this setup is 5-25; improvements would require attention mechanisms or transformer architectures.

### 3.5 - RNN Comparison Table

In [ ]:
df_cmp = pd.read_csv("results/rnn_comparison.csv")
print(df_cmp.to_string(index=False))

| Model | Context Memory | Complexity |
|-------|---------------|------------|
| RNN | Poor - vanishing gradients | Low |
| LSTM | Good - cell state preserves long-range info | High (4 gates) |
| GRU | Good - simpler than LSTM | Medium (2 gates) |
| Seq2Seq LSTM | Strong - fixed context vector from encoder | High |

### 3.6 - Synthesis

**Q: Why does the Seq2Seq model outperform a standalone LSTM on translation?**

A standalone LSTM processes the source and immediately tries to generate the target, requiring it to simultaneously read and write. The Seq2Seq architecture separates these roles: the **encoder** compresses the entire source sentence into a context vector (final hidden state), and the **decoder** attends exclusively to generating the target. This separation allows the encoder to build a complete source representation before any generation begins. The bottleneck is the fixed-size context vector - **attention mechanisms** (Bahdanau, 2015) remove this bottleneck by allowing the decoder to query all encoder hidden states dynamically, which is the foundation of modern transformers.

---
## Conclusion

This project demonstrated three core deep learning paradigms:

1. **MLP:** Effective for tabular data with careful normalisation, class weighting, and regularisation (dropout, weight decay). Initialisation strategy materially affects convergence speed.

2. **CNN:** Weight sharing and local receptive fields make CNNs the natural choice for spatial data. Architectural choices (padding, stride, pooling type, filter count) involve accuracy/efficiency trade-offs that must be empirically validated.

3. **RNN/LSTM/GRU/Seq2Seq:** Gated units solve the vanishing gradient problem. Seq2Seq with teacher forcing enables translation. Beam search improves generation over greedy decoding. The fixed context vector bottleneck motivates attention - the core of the transformer.

**Overall limitations:**
- All models are trained on CPU or a single GPU; larger datasets and architectures require distributed training.
- The Seq2Seq model lacks attention, which severely limits translation quality on long sentences.
- Fashion-MNIST accuracy can be pushed higher with residual connections, batch normalisation, and learning rate scheduling.